# Latent gate, Stage 1: train a model to use more latent steps under a cache budget

Fine-tunes SIM-CoT's CODI GPT-2 (or CODI) in the eight arms of `results/latent_gate/stage1/PLAN.md`, evaluates each on the 500
validation questions at 6, 12 and 24 latent steps, then applies the plan's decision rules. Idea:
`docs/LATENT_GATE_IDEA.md`.

Each arm's finished model and evaluation go to Drive, so **rerunning after a disconnect skips finished arms** and redoes
only the arm that was cut off. Every arm takes the same seeded 32,768 problems in the same order.

**Before running:**
- Free about 5 GB of Google Drive storage (each finished model is about 0.5 GB).
- Expect roughly 3 hours on an A100 for all arms and evaluations, longer on a T4. Reruns pick up where they stopped.
- Runtime → Change runtime type → **GPU** (A100 if offered; a T4 works, slower), then Runtime → **Run all**.
- Keep this tab open until the last cell, which downloads the results.

In [ ]:
# A run is identified by these settings; rerunning with the same values resumes it.
BASE = "simcot"          # "simcot": internlm/SIM_COT-GPT2-CODI; "codi": zen-E/CODI-gpt2 (the replication)
# gate12_label is left out: Stage 0 found the decode labels do not track which entries matter (see the plan).
ARMS = ["ft6", "ft6x2", "ft12", "ftrand", "window12", "rand12", "gate12"]
RECORDS = 32_768         # one pass at 32 per update = 1,024 updates
EVAL_LIMIT = None        # e.g. 50 for a quick look; None scores all 500 validation questions
RUN_TAG = ""             # e.g. "-seed2" with a different SEED for a second seed
SEED = 7
REPO_REF = "main"        # the branch or commit holding scripts/latent_gate

In [ ]:
from pathlib import Path
import json, os, runpy, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
RUN_NAME = f"latent-gate-{BASE}-n{RECORDS}-s{SEED}{RUN_TAG}"
BACKUP_DIR = DRIVE / "latent-gate" / RUN_NAME
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode:
    raise SystemExit("No GPU: Runtime > Change runtime type > GPU, then Runtime > Run all.")
print("GPU:", gpu.stdout.strip())

PINNED = BACKUP_DIR / "commit.txt"  # a started run keeps the code it started with
ref = PINNED.read_text().strip() if PINNED.exists() else REPO_REF
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/sttruji/dynamic-state-anchoring.git", str(REPO_DIR)],
                   check=True)
subprocess.run(["git", "fetch", "origin", ref], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert Path("scripts/latent_gate/stage1_train.py").exists(), f"{ref} has no scripts/latent_gate; push it first"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
PINNED.write_text(COMMIT)
print("Code:", COMMIT, f"(from {ref})")

In [ ]:
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")
os.environ["HF_HOME"] = str(DRIVE / "cache/huggingface")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
WORK = Path("/content/latent-gate")  # fast local disk; finished arms are copied to BACKUP_DIR
WORK.mkdir(exist_ok=True)
run_logged = runpy.run_path("scripts/colab_process.py")["run_logged"]

def run(*args):
    run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR, log_dir=BACKUP_DIR / "logs")

run("scripts/setup_data.py", "--eval-only")  # evaluation sets and the pinned GSM8K-Aug arrays, train included
CHECKPOINT = None
if BASE == "simcot":
    CHECKPOINT = DRIVE / "checkpoints/simcot-gpt2"
    run("scripts/latent_gate/fetch_simcot.py", "--output", CHECKPOINT)

In [ ]:
EVALS = BACKUP_DIR / "evals"
limit = [] if EVAL_LIMIT is None else ["--limit", EVAL_LIMIT]

# The frozen base gets every policy; each arm gets the plain cache plus the condition it was trained in.
POLICIES = {"base": [], "ft12": ["plain", "dedup_soft"], "window12": ["plain", "window"], "rand12": ["plain", "random"],
            "gate12": ["plain", "gate", "gate_soft"], "gate12_label": ["plain", "gate", "gate_soft"]}

def evaluate(name, model_dir):
    target = EVALS / name / "gsm8k_valid.jsonl"
    if target.exists() and EVAL_LIMIT is None:
        print(f"{name}: evaluation already on Drive")
        return
    local = WORK / "evals" / name / "gsm8k_valid.jsonl"
    run("scripts/latent_gate/stage1_eval.py", *([] if model_dir is None else ["--checkpoint", model_dir]),
        "--dataset", "gsm8k_valid", "--output", local, "--device", "cuda", *limit,
        *(["--policies", *POLICIES.get(name, ["plain"])] if POLICIES.get(name, ["plain"]) else []))
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local, target)

evaluate("base", CHECKPOINT)  # the frozen starting model, including the logn rival fix
for arm in ARMS:
    saved = BACKUP_DIR / "arms" / arm
    if not (saved / "model/pytorch_model.bin").exists():
        out = WORK / "arms" / arm
        run("scripts/latent_gate/stage1_train.py", "--arm", arm, *([] if CHECKPOINT is None else ["--checkpoint", CHECKPOINT]),
            "--output", out, "--records", RECORDS, "--seed", SEED, "--device", "cuda", "--bf16")
        shutil.copytree(out / "model", saved / "model", dirs_exist_ok=True)
        for name in ("args.json", "metrics.jsonl"):
            shutil.copy2(out / name, saved / name)
    else:
        print(f"{arm}: model already on Drive")
    evaluate(arm, saved / "model")

In [ ]:
run("scripts/latent_gate/stage1_report.py", EVALS)

In [ ]:
# The evaluations, training logs and arguments, laid out for the repo's results/latent_gate/stage1 folder.
from google.colab import files

bundle = Path(f"/content/{RUN_NAME}")
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(EVALS, bundle / "evals")
for arm in ARMS:
    for name in ("args.json", "metrics.jsonl"):
        source = BACKUP_DIR / "arms" / arm / name
        if source.exists():
            (bundle / "training" / arm).mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, bundle / "training" / arm / name)
(bundle / "commit.txt").write_text(COMMIT)
files.download(shutil.make_archive(str(bundle), "zip", bundle))
print(f"Downloading {bundle}.zip. If it does not start, get it from the Files panel on the left.")

**Reading the result**
- The report cell applies the plan's rules. **H2** is the main claim: the gated arm at 12 steps beats `ft6` at 6 on the
  173 problems with 4 or more rationale steps, without losing overall and without falling at 24.
- The rival rows show whether a simpler fix does as well: training at depth (`ft12`), random depth (`ftrand`), a fixed
  window (`window12`), training under random eviction (`rand12`), twice the 6-step compute (`ft6x2`), or the
  untrained `logn`, `logscale` and `dedup_soft` fixes.
- `gate12_label` is off by default: in Stage 0 the decode labels did not pick the entries that matter. Add it to
  `ARMS` only to test the label anyway, outside the plan.
- **Stop** means no 12-step arm beat `ft6` on the long problems; then GPT-2 does not use the extra depth.
- Held-out sets stay unscored until the comparison is settled.

**Sharing:** the last cell downloads a zip of the evaluations and training logs. Point Claude at it.